<a href="https://colab.research.google.com/github/DiogoBotton/LLM_Learning/blob/main/RAG/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introdução ao RAG

Casos de uso mais poderosas: Criação de chatbots sofisticados de perguntas e respostas (Q&A). Esses chatbots podem responder a perguntas sobre informações específicas usando RAG.

### Instalação das bibliotecas (uso no colab)

In [1]:
!pip install -q transformers einops accelerate bitsandbytes
!pip install -q langchain langchain_community langchain-huggingface langchainhub langchain_chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 23.3 MB/s eta 0:00

#### Importações

In [2]:
import torch
import os
import getpass

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

from langchain.prompts import PromptTemplate
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [3]:
# Para adquirir o token do HugginFace
os.environ["HF_TOKEN"] = getpass.getpass()

··········


### Carregando a LLM com Quantização

OBS. Quantização não funcionou localmente na máquina com WSL2 e nem baixar o modelo do Llama sem quantização, então faremos no Colab.

In [4]:
#model_id = "microsoft/Phi-3-mini-4k-instruct" # Modelo Phi-3
model_id = "meta-llama/Meta-Llama-3-8B-Instruct" # Modelo Llama 3

In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=500,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Device set to use cuda:0


### Template e Chain

Abaixo há os templates do Llama

In [6]:
# PHI 3
#template = """
#<|system|>
#Você é um assistente virtual prestativo e está respondendo perguntas gerais. <|end|>
#<|user|>
#{pergunta}<|end|>
#<|assistant|>
#"""

# LLAMA 3
template = """
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Você é um assistente virtual prestativo e está respondendo perguntas gerais.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{pergunta}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

template

'\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{pergunta}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'

In [7]:
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{pergunta}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n')

Abaixo é um contexto bem interessante para utilizar RAG, pois está claro que o modelo não tem como saber nem mesmo que dia é hoje.

In [8]:
chain = prompt | llm

# Para preencher a variavel do template "pergunta" necessário definir como um dicionario na função invoke
output = chain.invoke({"pergunta": "Que dia é hoje?"})
print(output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Peço desculpas, mas como sou um assistente virtual, não tenho acesso a informações em tempo real sobre a data atual. No entanto, posso ajudá-lo com outras coisas!


### Prompt para RAG

Para implementar o RAG devemos reservar um espaço no template do prompt para que seja alocado o contexto que queremos usar.

No prompt do sistema (system_prompt) é importante colocar que se caso o modelo não saiba a resposta, apenas dizer que não sabe, isso ajuda o modelo a não alucinar.

Caso for necessário, também é interessante adicionar na TAG de sistema: **"Responda a pergunta com base apenas no contexto"** caso as respostas sejam muito alucinadas. Pode acontecer da LLM fazer uma combinação do que ela já sabe com o contexto e gerar uma informação que pode estar errada.

In [9]:
template_rag = """
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Você é um assistente virtual prestativo e está respondendo perguntas gerais.
Use os seguintes pedaços de contexto recuperado para responder à pergunta.
Se você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Pergunta: {pergunta}
Contexto: {contexto}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [10]:
prompt_rag = PromptTemplate.from_template(template_rag)
prompt_rag

PromptTemplate(input_variables=['contexto', 'pergunta'], input_types={}, partial_variables={}, template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\nUse os seguintes pedaços de contexto recuperado para responder à pergunta.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n')

### Definindo o contexto

Para resolver o problema do modelo não saber que dia é hoje iremos definir o contexto.

In [11]:
from datetime import date

dia = date.today()
print(dia)

2025-10-30


In [12]:
contexto = f"Você sabe que hoje é dia {dia}"
print(contexto)

Você sabe que hoje é dia 2025-10-30


Agora a LLM, a partir do contexto, sabe a data de hoje.

In [13]:
chain_rag = prompt_rag | llm | StrOutputParser()

pergunta = "Que dia é hoje? Retorne a data em formato dd/mm/yyyy"
res = chain_rag.invoke({"pergunta": pergunta, "contexto": contexto})
print(res)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Hoje é 30/10/2025.


### RAG - Explorando mais

Vamos supor que precisamos usar a LLM para responder dúvidas referentes a um documento ou planilha que contém informações de uma empresa. Esses dados são privados e os modelos não tem como saber. Mesmo se os dados fossem publicos, com o tempo esses dados não seriam totalmente recentes.

Por enquanto, adicionaremos os dados via código antes de pegarmos externamente.

É possível tratar e enviar dados de uma planilha, por exemplo, para a LLM ter contexto para responder certas dúvidas.

In [14]:
contexto = """Faturamento trimestral:
1º: R$42476,40
2º: R$46212,97
3º: R$41324,56
4º: R$56430,24"""

pergunta = "Qual trimestre teve o maior faturamento?"
res = chain_rag.invoke({
    "pergunta": pergunta,
    "contexto": contexto
})
print(res)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


O trimestre com o maior faturamento foi o 4º, com R$56430,24.


### Depuração / Debugging

Há tres tipos de depuração:
- Modo Debug: Isso adiciona instruções de registro para todos os eventos em sua cadeia.
- Modo Verbose: Isso adiciona instruções de impressão para eventos "importantes" em sua cadeia.
- Rastreamento com LangSmith: Isso registra eventos no LangSmith para um rastreamento melhor para cada uma das etapas.

In [15]:
from langchain.globals import set_debug
set_debug(True) # Ativando o debug

In [16]:
pergunta = "Qual trimestre teve o menor faturamento?"
res = chain_rag.invoke({
    "pergunta": pergunta,
    "contexto": contexto
})
print(res)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "pergunta": "Qual trimestre teve o menor faturamento?",
  "contexto": "Faturamento trimestral:\n1º: R$42476,40\n2º: R$46212,97\n3º: R$41324,56\n4º: R$56430,24"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "pergunta": "Qual trimestre teve o menor faturamento?",
  "contexto": "Faturamento trimestral:\n1º: R$42476,40\n2º: R$46212,97\n3º: R$41324,56\n4º: R$56430,24"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:HuggingFacePipeline] Entering LLM run with input:
{
  "prompts": [
    "<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\nUse os seguintes pedaços de contexto recuperado para responder à pergunta.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a 

In [17]:
set_debug(False) # Desativando o debug

## Aplicação para RAG com contextos maiores

### Etapas de indexação

#### 1 - Carregar o conteúdo

Nexte exemplo, o contexto será um texto relacionado a um artigo sobre o Oscar.

https://www.bbc.com/portuguese/articles/cd19vexw0y1o

In [18]:
from langchain_community.document_loaders import WebBaseLoader # Usa a biblioteca URLLib do python para carregar HTML de URLs
import bs4 # Bealtifulsoup4 faz a conversão do html carregado pelo WebBaseLoader para o formato de texto
from langchain_huggingface import HuggingFaceEmbeddings # Conversão dos textos para vetores
from langchain_chroma import Chroma

In [19]:
# É possível adicionar vários links
loader = WebBaseLoader(web_paths= ("https://www.bbc.com/portuguese/articles/cd19vexw0y1o",))
docs = loader.load()

In [20]:
print(len(docs[0].page_content)) # Quantidade de tokens

12190


In [21]:
print(docs[0].page_content[:100]) # Visualizando os primeiros 100 tokens

Oscar 2024: confira todos os ganhadores dos prêmios da Academia de Hollywood  - BBC News BrasilBBC N


### 2 - Divisão em pedaços de texto / Split

Esse documento tem mais de 10 mil caracteres, algo que seria muito grande para um contexto.

Para certos modelos o tamanho da janela de contexto é cerca de 8 mil tokens, o que equivale a 32 mil caracteres. Até que para esse nosso caso específico seria possível encaixar o texto dentro do contexto da LLM, porém, geralmente as LLMs terão dificuldades em localizar informações dentro de estradas tão longas, por isso iremos realizar o processo de divisão.

`chunk_overlap`: Significa sobreposição. Ajuda a evitar que uma declaração seja separada do seu contexto importante ao realizar a divisão (split) do texto. Uma sobreposição de chunks maiores resultará em mais pedaços compartilhando caracteres comuns, enquanto uma sobreposição de chunks menores resultará em menos pedaços compartilhando caracteres comuns.

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # Dividirá recursivamente o documento
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, # Dividirá o texto em pedaços de 1000 caracteres
                                               chunk_overlap = 200,
                                               add_start_index = True) # Para o índice dos caracteres inicial seja preservado

In [23]:
splits = text_splitter.split_documents(docs)

In [24]:
len(splits)

15

Em cada divisão os metadados também são mostrados, como a fonte (source), titulo, descrição e assim por diante.

In [25]:
splits[0] # Exibindo a primeira divisão de dados

Document(metadata={'source': 'https://www.bbc.com/portuguese/articles/cd19vexw0y1o', 'title': 'Oscar 2024: confira todos os ganhadores dos prêmios da Academia de Hollywood  - BBC News Brasil', 'description': "'Oppenheimer' foi o grande vencedor da noite com sete estatuetas, incluindo o prêmio de melhor filme, melhor diretor e melhor ator.", 'language': 'pt-br', 'start_index': 0}, page_content="Oscar 2024: confira todos os ganhadores dos prêmios da Academia de Hollywood  - BBC News BrasilBBC News, BrasilVá para o conteúdoSeçõesNotíciasBrasilInternacionalEconomiaSaúdeCiênciaTecnologiaVídeosPodcastsNotíciasBrasilInternacionalEconomiaSaúdeCiênciaTecnologiaVídeosPodcastsOscar 2024: confira todos os ganhadores dos prêmios da Academia de HollywoodCrédito, Getty ImagesLegenda da foto, Robert Downey Jr., Da'Vine Hoy Randolph, Emma Stone e Cillian Murphy com suas respectivas estatuetas do OscarArticle InformationAuthor, Leire VentasRole,  Correspondente da BBC News Mundo em Los AngelesX, @leire_

### 3 - Armazenamento

Precisamos indexar os pedaços de textos para que posteriormente sejam pesquisados. A maneira mais comum de fazer isso é incorporar o conteúdo de cada divisão de document e inserir esses embeddings em um banco de dados de vetores.

Quando queremos pesquisar em nossas divisões, pegamos uma consulta de pesquisa de texto, a incorporamos e realizamos algum tipo de pesquisa de "similaridade" para identificar as divisões armazenadas com os embeddings mais semelhantes ao nosso embedding de consulta (que seria o prompt do usuário).

#### Embeddings

Embedding é uma representação numérica de um texto. Olhando para eles, não são nada além de números, mas por trás dos panos, eles possuem uma relação entre si. Isso significa que palavras ou frases com **significados semelhantes** terão embeddings próximos uns dos outros em um espaço vetorial. Através dos embeddings conseguiremos procurar por **itens simulares** (nesse caso, palavras).

Podemos incorporar e armazenar todas as nossas divisões de documentos em um único comando usando o armazenamento de vetores Chroma, com o LangChain.

Existem vários modelos de embedding open source e proprietários para realizar esse processo, iremos utilizar o `sentence-transformers/all-mpnet-base-v2` (open source).

In [26]:
hf_embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
input_test = "Um teste apenas para testar o algoritmo de embedding"
result = hf_embeddings.embed_query(input_test) # Realizará vários cálculos matemáticos para transformar o texto em um vetor

In [28]:
len(result) # Foi convertido em um vetor com 768 posições

768

In [29]:
print(result)

[-0.019974304363131523, -0.04652781784534454, -0.03760574758052826, 0.043524835258722305, 0.01760284975171089, 0.003018475603312254, 0.02907024510204792, 0.039617374539375305, 0.04744008556008339, 0.002828981727361679, 0.03352710232138634, 0.026037264615297318, 0.015550483018159866, -0.0009583882638253272, 0.028968743979930878, -0.042637038975954056, -0.02726990357041359, 0.03916770592331886, -0.05789683759212494, -0.028128748759627342, -0.016727088019251823, -0.04455960914492607, 0.005685081239789724, 0.006812114734202623, 0.01664028875529766, 0.019474413245916367, -0.0036274162121117115, -0.03879229351878166, 0.005641467869281769, -0.030780982226133347, -0.0024974094703793526, -0.024469399824738503, 0.05260040983557701, -0.015682194381952286, 1.849382556429191e-06, -0.03460341691970825, -0.056726280599832535, 0.00245489040389657, 0.010813231579959393, -0.022237705066800117, 0.031676262617111206, 0.044152434915304184, -0.001390016172081232, -0.021642398089170456, -0.05626467242836952,

#### Armazenando no banco de dados vetorial

Utilizaremos o banco de dados vetorial Chroma, um dos mais utilizados e versateis.

No celula de código abaixo é realizado o embedding na base de dados (splits) e armazenamento no formato de vetor. Basicamente esta etapa conclui o processo de indexação da pipeline de RAG.

Até este ponto, já temos um repositório de vetores que pode ser consultado pela LLM.

In [30]:
vector_store = Chroma.from_documents(
    documents=splits, # O texto com as divisões
    embedding=hf_embeddings) # Algoritmo de embedding

vector_store

### Etapas de Recuperação e geração

### 4 - Configurando o recuperador de texto / Retriever

Quanto maior a base de dados, maior deve ser o número de K `search_kwargs` para que sejam retornados mais documentos. O valor 6 é considerado um valor padrão. Porém, é interessante realizar o teste com mais ou menos valores de acordo com o caso.

In [31]:
# Busca baseada em similaridade entre os embeddings
retriever = vector_store.as_retriever(search_type = "similarity",
                                      search_kwargs={"k": 6}) # Limita o número de documentos retornados pelo recuperador (k vizinhos próximos)

### 5 - Geração

In [32]:
# Iremos utilizar o template abaixo, que enviará a pergunta juntamente com o contexto
template_rag

'\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\nUse os seguintes pedaços de contexto recuperado para responder à pergunta.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'

Criando o prompt.

In [33]:
prompt_rag = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template=template_rag
)

prompt_rag

PromptTemplate(input_variables=['contexto', 'pergunta'], input_types={}, partial_variables={}, template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está respondendo perguntas gerais.\nUse os seguintes pedaços de contexto recuperado para responder à pergunta.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n')

Formatação de documentos

A função abaixo recebe uma lista de documentos e retorna uma única string, onde o conteúdo de cada documento é concatenado com duas quebras de linha entre eles. No contexto de processamento de texto para uma pipeline de RAG, essa função fará a formatação dos documentos recuperados, de modo que o seu conteúdo possa ser passado de maneira estruturada para um modelo de linguagem, facilitando a geração de respostas baseadas nos dados dos documentos.

In [35]:
# Função para ajudar na visualização dos dados
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [36]:
chain_rag = ({"contexto": retriever | format_docs, "pergunta": RunnablePassthrough()}
             | prompt_rag
             | llm
             | StrOutputParser())

In [34]:
# Teste sem RAG
chain.invoke("Qual filme ganhou mais oscars na premiação de 2024?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'Infelizmente, a premiação dos Oscars de 2024 ainda não ocorreu. A cerimônia anual dos Academy Awards é realizada todos os anos em fevereiro ou março, e o resultado das votações é divulgado apenas após a premiação. Portanto, não há um filme que tenha ganhado mais Oscars na premiação de 2024, pois essa premiação ainda não ocorreu.'

In [37]:
# Teste com RAG
chain_rag.invoke("Qual filme ganhou mais oscars na premiação de 2024?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'Segundo o texto, o filme "Oppenheimer" ganhou sete estatuetas, incluindo o prêmio de melhor filme, melhor diretor (Christopher Nolan), melhor ator (Cillian Murphy) e melhor ator coadjuvante (Robert Downey Jr.). Portanto, o filme "Oppenheimer" é o grande vencedor da 96ª edição do Oscar.'

Caso seja necessário, é possível deletar o contexto com o comando abaixo.

O que pode ser necessário durante redefinições, atualizações. Por exemplo, excluir dados antigos para dar espaço para novos dados a serem processados e armazenados.

In [ ]:
vector_store.delete_collection()